# 10. 모듈 / 패키지 구조화

`import`의 여러 형태, 패키지에서 `__init__.py`의 역할, `if __name__ == "__main__":` 관용구, 상대/절대 import 차이를 다룹니다.

## import의 여러 형태

In [1]:
import math                  # 모듈 전체를 가져와서 math.sqrt 처럼 사용
from math import sqrt, pi    # 특정 이름만 가져와서 바로 사용
import math as m             # as로 별칭 부여 (관례: numpy -> np, pandas -> pd 식으로 널리 쓰임)
from math import *           # 모든 이름을 가져옴 (이름 충돌 위험 때문에 실무에서는 지양)

print(math.sqrt(16))
print(sqrt(16), pi)
print(m.sqrt(16))


4.0
4.0 3.141592653589793
4.0


## 실제로 모듈 파일 하나 만들어서 import 해보기

노트북 안에서는 바로 다른 `.py` 파일을 만들어볼 수 있습니다 — `import`가 실제로 파일을 찾는 과정을 체험해봅니다.

In [2]:
import os, tempfile, sys

workdir = tempfile.mkdtemp()
sys.path.insert(0, workdir)  # 파이썬이 모듈을 찾는 경로 목록에 추가

with open(os.path.join(workdir, "greetings.py"), "w") as f:
    f.write('''
def hello(name):
    return f"안녕, {name}"

GREETING_COUNT = 0
''')

import greetings
print(greetings.hello("Noni"))
print(greetings.GREETING_COUNT)


안녕, Noni
0


## `if __name__ == "__main__":`

모듈이 **직접 실행됐을 때만** 특정 코드를 돌리고 싶을 때 씁니다. 다른 파일에서 `import`만 할 때는 실행되지 않습니다.

In [3]:
script_content = '''
def main():
    print("이 스크립트를 직접 실행했을 때만 보임")

if __name__ == "__main__":
    main()
'''

with open(os.path.join(workdir, "runner.py"), "w") as f:
    f.write(script_content)

import runner  # import만 하면 main()이 실행되지 않음 (출력 없음)
print("import만 했을 때는 위에 아무것도 안 찍힘 (의도된 동작)")

runner.main()  # 명시적으로 호출해야 실행됨


import만 했을 때는 위에 아무것도 안 찍힘 (의도된 동작)
이 스크립트를 직접 실행했을 때만 보임


## 패키지와 `__init__.py`

폴더에 `__init__.py`가 있으면 파이썬이 그 폴더를 '패키지'로 인식합니다. 패키지 레벨에서 무엇을 노출할지 정리하는 역할도 합니다.

In [4]:
pkg_dir = os.path.join(workdir, "mlpkg")
os.makedirs(pkg_dir, exist_ok=True)

with open(os.path.join(pkg_dir, "__init__.py"), "w") as f:
    f.write("from .model import predict\n")  # 패키지 바로 아래서 predict를 쓸 수 있도록 노출

with open(os.path.join(pkg_dir, "model.py"), "w") as f:
    f.write("def predict(x):\n    return x * 2\n")

import mlpkg
print(mlpkg.predict(21))  # mlpkg.model.predict가 아니라 mlpkg.predict로 바로 접근


42


## 절대 import vs 상대 import (개념 설명)

패키지 **내부** 모듈끼리는 상대 import(`from . import model`, `from ..utils import helper`)를 쓸 수 있습니다. 노트북 환경에서는 직접 실행하기 까다로워서(상대 import는 패키지로 실행될 때만 동작) 개념만 짚고 넘어갑니다.

- 절대 import: `from mlpkg.model import predict` — 프로젝트 루트 기준 전체 경로
- 상대 import: `from .model import predict` — 같은 패키지 내에서 상대적 위치 기준

실무 규칙: 보통 패키지 내부에서는 상대 import, 패키지 밖에서 그 패키지를 쓸 때는 절대 import를 사용합니다.

## 헷갈렸던 점 / 더 찾아볼 것

- `sys.path`에 뭐가 들어있는지, 왜 `ModuleNotFoundError`가 나는지는 결국 이 경로 목록 문제인 경우가 많음
- 패키지를 pip로 설치 가능하게 만들려면 `pyproject.toml`/`setup.py` 쪽을 따로 찾아볼 것